In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor
)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [2]:
DATA_PATH = Path("../data/processed")

train_data = pd.read_csv(DATA_PATH / "train_data.csv")
valid_data = pd.read_csv(DATA_PATH / "valid_data.csv")

train_data["date"] = pd.to_datetime(train_data["date"])
valid_data["date"] = pd.to_datetime(valid_data["date"])

In [3]:
SAMPLE_SIZE = 200_000

train_sample = train_data.sample(
    n=SAMPLE_SIZE,
    random_state=42
)

In [4]:
print(train_sample.shape)
print(valid_data.shape)

(200000, 42)
(167508, 42)


In [5]:
TARGET = "sales"

X_train = train_sample.drop(columns=[TARGET])
y_train = train_sample[TARGET]

X_valid = valid_data.drop(columns=[TARGET])
y_valid = valid_data[TARGET]

In [6]:
X_train = X_train.drop(columns=["date"])
X_valid = X_valid.drop(columns=["date"])

In [7]:
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_columns = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_columns
        )
    ],
    remainder="passthrough"
)

In [9]:
def evaluate_model(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    wape = (
        np.abs(y_true - y_pred).sum()
        /
        np.abs(y_true).sum()
    )

    smape = (
        100
        *
        np.mean(
            (
                2*np.abs(y_true-y_pred)
            )
            /
            (
                np.abs(y_true)
                +
                np.abs(y_pred)
                +
                1e-10
            )
        )
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "WAPE": wape,
        "SMAPE": smape
    }

In [10]:
baseline_prediction = X_valid["lag_1"]

baseline_metrics = evaluate_model(
    y_valid,
    baseline_prediction
)

baseline_metrics

{'MAE': 124.01255151905046,
 'RMSE': np.float64(461.7674651514353),
 'R2': 0.8812808759081001,
 'WAPE': np.float64(0.25663661015004857),
 'SMAPE': np.float64(43.881707325685674)}

In [11]:
models = {

    "Linear Regression":
        LinearRegression(),

    "Ridge":
        Ridge(),

    "Lasso":
        Lasso(),


    "XGBoost":
        XGBRegressor(
            n_estimators=50,
            learning_rate=0.1,
            max_depth=4,
            random_state=42,
            n_jobs=-1
        ),

    "LightGBM":
        LGBMRegressor(
            n_estimators=50,
            learning_rate=0.1,
            random_state=42,
            verbose=-1
        ),

    "CatBoost":
        CatBoostRegressor(
            iterations=50,
            learning_rate=0.1,
            random_seed=42,
            verbose=0
        )
}

In [12]:
import time

In [13]:
print(X_train.shape)
print(X_valid.shape)
print(X_train.memory_usage(deep=True).sum() / 1024**2)

(200000, 40)
(167508, 40)
145.92648220062256


In [14]:
results = []

trained_models = {}

for model_name, model in models.items():

    print("="*60)
    print(f"Training : {model_name}")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    train_start = time.perf_counter()

    pipeline.fit(
        X_train,
        y_train
    )

    train_time = time.perf_counter() - train_start

    predict_start = time.perf_counter()

    predictions = pipeline.predict(
        X_valid
    )

    prediction_time = time.perf_counter() - predict_start

    metrics = evaluate_model(
        y_valid,
        predictions
    )

    results.append({

        "Model": model_name,

        "MAE": metrics["MAE"],

        "RMSE": metrics["RMSE"],

        "R2": metrics["R2"],

        "WAPE": metrics["WAPE"],

        "SMAPE": metrics["SMAPE"],

        "Train Time": round(train_time,2),

        "Prediction Time": round(prediction_time,2)

    })

    trained_models[model_name] = pipeline

    print("Completed")

Training : Linear Regression
Completed
Training : Ridge
Completed
Training : Lasso
Completed
Training : XGBoost
Completed
Training : LightGBM
Completed
Training : CatBoost
Completed


In [15]:
results = pd.DataFrame(results)

results = results.sort_values(
    by="WAPE"
).reset_index(drop=True)

results

,Model,MAE,RMSE,R2,WAPE,SMAPE,Train Time,Prediction Time
0,LightGBM,70.617915,243.446272,0.967003,0.146140,61.972958,16.06,1.61
1,XGBoost,75.957672,263.560450,0.961325,0.157190,64.201551,4.83,2.30
2,CatBoost,82.423775,266.809547,0.960365,0.170571,72.136357,6.14,1.74
3,Ridge,90.625688,312.276541,0.945706,0.187544,77.671066,1.10,0.64
4,Linear Regression,91.801368,316.010880,0.944400,0.189977,86.097556,1.76,0.76
5,Lasso,101.063681,311.356689,0.946025,0.209145,100.748336,40.51,2.06


In [16]:
best_model_name = results.iloc[0]["Model"]

best_pipeline = trained_models[
    best_model_name
]

print(best_model_name)

LightGBM


In [17]:
best_model_name = results.iloc[0]["Model"]

best_pipeline = trained_models[
    best_model_name
]

print(best_model_name)

LightGBM


In [18]:
import joblib
MODEL_PATH = Path("../artifacts")

MODEL_PATH.mkdir(exist_ok=True)

joblib.dump(
    best_pipeline,
    MODEL_PATH / "best_pipeline.pkl"
)

['..\\artifacts\\best_pipeline.pkl']

In [19]:
loaded_pipeline = joblib.load(
    MODEL_PATH / "best_pipeline.pkl"
)

loaded_pipeline.predict(
    X_valid.head()
)

array([7.80299434, 7.80299434, 7.80299434, 7.80299434, 7.80299434])

## HyperParameters Tunning Starts

In [20]:
from sklearn.model_selection import RandomizedSearchCV

In [21]:
param_grid = {

    "model__n_estimators": [50, 100, 150],

    "model__learning_rate": [0.01, 0.05, 0.1],

    "model__max_depth": [4, 6, 8],

    "model__num_leaves": [31, 63, 127],

    "model__subsample": [0.8, 1.0],

    "model__colsample_bytree": [0.8, 1.0]
}

In [22]:
lightgbm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMRegressor(
        random_state=42,
        verbose=-1
    ))
])

In [23]:
random_search = RandomizedSearchCV(
    estimator=lightgbm_pipeline,
    param_distributions=param_grid,
    n_iter=10,
    scoring="neg_mean_absolute_error",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

In [24]:
random_search.fit(
    X_train,
    y_train
)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


,estimator,Pipeline(step...verbose=-1))])
,param_distributions,"{'model__colsample_bytree': [0.8, 1.0], 'model__learning_rate': [0.01, 0.05, ...], 'model__max_depth': [4, 6, ...], 'model__n_estimators': [50, 100, ...], ...}"
,n_iter,10
,scoring,'neg_mean_absolute_error'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [25]:
print(random_search.best_params_)

{'model__subsample': 0.8, 'model__num_leaves': 63, 'model__n_estimators': 150, 'model__max_depth': 8, 'model__learning_rate': 0.05, 'model__colsample_bytree': 1.0}


In [26]:
best_lightgbm = random_search.best_estimator_

In [27]:
predictions = best_lightgbm.predict(X_valid)

tuned_metrics = evaluate_model(
    y_valid,
    predictions
)

pd.DataFrame([tuned_metrics])

,MAE,RMSE,R2,WAPE,SMAPE
0,66.969534,235.446834,0.969135,0.138589,56.145772
